In [29]:
import numpy as np

**Standard Inequality QP:**

$\underset{x}{\min} \quad \frac{1}{2} x^\top G x + x^\top c$ 

$\text{s.t.} \quad Ax \geq b$

$ x, c \in \mathbb{R}^n$

$ b \in \mathbb{R}^m$

$ A \in \mathbb{R}^{m\times n}$

$ G \in \mathbb{S}^{n\times n}$

KKT System (16.58):

$$
\underbrace{\begin{bmatrix}
G & 0 & -A^T \\
A & -I & 0 \\
0 & \Lambda & \mathcal{Y}
\end{bmatrix}}_{(2m+n)\times(2m+n)}
\underbrace{\begin{bmatrix}
\Delta x \\
\Delta y \\
\Delta \lambda
\end{bmatrix}}_{(2m+n)\times1}
=
\underbrace{\begin{bmatrix}
-r_d \\
-r_p \\
-\Lambda \mathcal{Y} e + \sigma \mu e
\end{bmatrix}}_{(2m+n)\times1}
$$

In [30]:
class Problem:
    def __init__(self, x, c, b, A, G, y, l):
        self.x = x
        self.n = x.shape[0]
        self.c = c
        self.b = b
        self.m = b.shape[0]
        self.A = A
        self.G = G
        self.y = y
        self.l = l

        # variables that are used in solver, there's probably a better way to store them
        self.dx_aff = np.zeros(self.n)
        self.dy_aff = np.zeros(self.m)
        self.dl_aff = np.zeros(self.m)
        self.dx = np.zeros(self.n)
        self.dy = np.zeros(self.m)
        self.dl = np.zeros(self.m)

    def create_KKT_sys(self, sigma, mu, eq):
        LHS_row1 = [self.G, np.zeros([self.n, self.m]), -self.A.T]
        LHS_row2 = [self.A, -np.eye(self.m), np.zeros([self.m, self.m])]
        LHS_row3 = [np.zeros([self.m, self.n]), np.diag(self.l), np.diag(self.y)]

        LHS = np.block([LHS_row1, LHS_row2, LHS_row3])

        r_d = self.G @ self.x - self.A.T @ self.l + self.c
        r_p = self.A @ self.x - self.y - self.b

        # can also just subtract -(dl_aff * dy_aff) from third row
        match eq:
            case '58':
                RHS_row3 = -(self.l * self.y)
            case '67':
                RHS_row3 = -(self.l * self.y) - (self.dl_aff * self.dy_aff) + sigma * mu

        RHS = np.block([-r_d, -r_p, RHS_row3])

        return LHS, RHS

<img src="images/algorithm.png" width="600">

In [31]:
class Solver:
    def __init__(self, problem):
        self.problem = problem

    def initial_value(self):

        return (x_0, y_0, l_0)

    def calc_ahat_aff(self):

        return ahat_aff

    def choose_tau_k(self, step):

        return tau_k

    def calc_ahat(self, tau_k):

        return ahat

    def take_step(self):
        LHS_aff, RHS_aff = self.problem.create_KKT_sys(sigma=0, mu=0, eq='58')

        self.problem.dx_aff, self.problem.dy_aff, self.problem.dl.aff = np.linalg.solve(LHS_aff, RHS_aff)

        mu = self.problem.y @ self.problem.l / self.problem.m

        ahat_aff = self.calc_ahat_aff()

        mu_aff = (self.problem.y + ahat_aff * self.problem.dy_aff).T @ ...
        (self.problem.l + ahat_aff * self.problem.dl_aff) / self.problem.m

        sigma = (mu_aff / mu)^3

        LHS_k, RHS_k = self.problem.create_KKT_sys(sigma, mu, eq='67')

        self.problem.dx, self.problem.dy, self.problem.dl = np.linalg.solve(LHS_k, RHS_k)

        tau_k = self.choose_tau_k()

        ahat = self.calc_ahat(self.problem, tau_k)

        x_kp1 = self.problem.x + ahat * self.problem.dx
        y_kp1 = self.problem.y + ahat * self.problem.dy
        l_kp1 = self.problem.l + ahat * self.problem.dl

        return x_kp1, y_kp1, l_kp1

    def solve_problem(self):
        for k in self.steps:
            self.take_step()

        return sol

In [32]:
n = 3
m = 2

x = np.ones(n)
c = np.ones(n)

b = np.ones(m)

A = np.ones([m, n])
G = np.ones([n, n])

y = np.ones(m)
l = np.ones(m)

problem = Problem(x, c, b, A, G, y, l)

LHS, RHS = problem.create_KKT_sys(sigma=0.9, mu=0.9, eq='58')